####Handling Null values
1. Equality check
2. Null in expressions
3. [Conditional functions for Null](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html#conditional-functions)
4. Null in aggregations

####1. Create a data frame to demo some scenarios

In [0]:
from pyspark.sql import SparkSession

spark_session = SparkSession.builder.getOrCreate()

In [0]:
person_list = [(100, "Prashant", 30),
             (101, "David", None),
             (102, "Sushant",None),
             (103, "Abdul", 45),
             (104, "Shruti", 28)]

person_df = spark_session.createDataFrame(person_list).toDF("id","name","age")
display(person_df)

####2. How null equality is executed.

2.1 Find all records where age is 28.\
2.1 Find all records where age is not given or unknown.

In [0]:
person_df.where("age == 28").display()

In [0]:
person_df.where("age == null").display()

2.2 Create a boolean column to investigate

In [0]:
from pyspark.sql.functions import expr
person_df.withColumn("age_null_expr", expr("age == null")).display() #Basic debugging technique created a column to check what == null evaluate so basically as output its always null unknown values produces unknown values so null with equality will always return null.



In [0]:
person_df.withColumn("age_null_expr", expr("age IS null")).display()
person_df.withColumn("age_null_expr", expr("age IS NOT null")).display()

#if you want to check null use IS or IS NOT instead of == or !=


2.3 Use case for null equality

Select only those persons having a valid age information

In [0]:
person_df.where("age IS null").display()

####3. How operators work on null values

3.1 Check the result of > operator on null values

In [0]:
person_df.withColumn("age_gt_29", expr("age>29")).display() 
#null will not participate in any comparison operation

3.2 How the comperison operator behaves in answering business questions.

Find all employees where age is greater than 29

In [0]:
person_df.where("age > 29").display()

3.3 How mathametical operators work on null.\
Calculate experience for every employee using the following formula.\
experience = age - 23


In [0]:
person_df.withColumn("experience",expr("age - 23")).display()

3.4. Calculate the experience knowing that age could be null.\
If age is null then assume 23 years for experience calculation.

In [0]:
person_df.withColumn("experience",expr("nvl(age,23) - 23")).display()

####4. How aggregates work on null values

4.1 What is the average age?

In [0]:
person_df.selectExpr("avg(age)").display()
# Aggeragate filter out null than calculate the avg

3.2 What if we filter out null values before aggregation

In [0]:
person_df.where("age IS NOT NULL").selectExpr("avg(age)").display()